In [1]:
import requests
import json
from typing import Dict, List, Any, Optional

def fetch_who_measure_data(indicator_code: str, country_groups: List[str] = None, countries: List[str] = None) -> Dict[str, Any]:
    """
    Fetch data from WHO European Health Information Gateway API for a specific measure
    
    Args:
        indicator_code: The measure code (e.g., 'hfa_43')
        country_groups: List of country group codes (e.g., ['WHO_EURO', 'EU_MEMBERS'])
        countries: List of country ISO codes (e.g., ['ITA', 'MDA'])
    
    Returns:
        Parsed JSON data from the 'data' field, or empty dict if error
    """
    base_url = "https://dw.euro.who.int/api/v3/measures/"
    url = f"{base_url}{indicator_code}"
    
    # Build filter parameters
    filter_parts = []
    
    if country_groups:
        country_groups_str = ",".join(country_groups)
        filter_parts.append(f"COUNTRY_GRP:{country_groups_str}")
    
    if countries:
        countries_str = ",".join(countries)
        filter_parts.append(f"COUNTRY:{countries_str}")
    
    # Manually construct URL to avoid automatic encoding of colons
    if filter_parts:
        filter_string = ";".join(filter_parts)
        url = f"{url}?filter={filter_string}"
    
    # Create browser-like headers to avoid 403 errors
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.9',
        'Referer': 'https://gateway.euro.who.int/',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
        'Cache-Control': 'max-age=0',
        'Sec-Fetch-Dest': 'document',
        'Sec-Fetch-Mode': 'navigate',
        'Sec-Fetch-Site': 'none'
    }
    
    try:
        # Create a session to maintain cookies from the beginning
        session = requests.Session()
        
        # First access the main WHO Data Warehouse site to establish session
        main_url = "https://dw.euro.who.int"
        print(f"Establishing session with {main_url}...")
        session.get(main_url, headers=headers, timeout=30)
        
        # Then make the API request using the established session
        print(f"Making API request to {url}...")
        response = session.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        json_data = response.json()
        
        # Return the 'data' field from the JSON response
        return json_data.get("data", {})
        
    except requests.exceptions.RequestException as e:
        print(f"API request failed: {e}")
        return {}
    except json.JSONDecodeError as e:
        print(f"JSON parsing failed: {e}")
        return {}
    except Exception as e:
        print(f"Unexpected error: {e}")
        return {}

# Example usage:
# data = fetch_who_measure_data("hfa_43", ["WHO_EURO", "EU_MEMBERS"], ["ITA", "MDA"])

In [2]:
def parse_who_data_structure(data: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Parse and format WHO API data structure for easier analysis
    
    Args:
        data: Raw data from WHO API (json_data["data"])
    
    Returns:
        List of formatted data records
    """
    if not data:
        return []
    
    # Handle different possible data structures
    if isinstance(data, list):
        return data
    elif isinstance(data, dict):
        # If data is a dict, look for common keys that contain the actual data
        for key in ["values", "observations", "data", "records"]:
            if key in data and isinstance(data[key], list):
                return data[key]
        # If no list found, return the dict as a single item
        return [data]
    
    return []

def parse_who_json_to_dataframe(json_data: Dict[str, Any]) -> 'pd.DataFrame':
    """
    Parse WHO API JSON response and extract data into a pandas DataFrame
    
    Args:
        json_data: Raw JSON response from WHO API (can be dict or list)
    
    Returns:
        pandas DataFrame with extracted data
    """
    import pandas as pd
    
    if not json_data:
        return pd.DataFrame()
    
    # Handle both dict and list inputs
    if isinstance(json_data, list):
        # If json_data is already a list, use it directly
        data_records = json_data
    elif isinstance(json_data, dict):
        # If json_data is a dict, extract the data array
        data_records = json_data.get("data", [])
    else:
        return pd.DataFrame()
    
    if not data_records:
        return pd.DataFrame()
    
    # List to store flattened records
    flattened_records = []
    
    for record in data_records:
        if not isinstance(record, dict):
            continue
            
        # Extract dimensions (metadata about the record)
        dimensions = record.get("dimensions", {})
        
        # Extract value information
        value_info = record.get("value", {})
        
        # Create a flattened record
        flattened_record = {
            # Extract dimension fields
            "COUNTRY": dimensions.get("COUNTRY", ""),
            "COUNTRY_GRP": dimensions.get("COUNTRY_GRP", ""),
            "SEX": dimensions.get("SEX", ""),
            "YEAR": dimensions.get("YEAR", ""),
            
            # Extract value fields
            "VALUE_DISPLAY": value_info.get("display", ""),
            "VALUE_NUMERIC": value_info.get("numeric", None),
            
            # Keep original record for reference
            "RAW_RECORD": record
        }
        
        # Add any other dimension fields that might exist
        for key, value in dimensions.items():
            if key not in ["COUNTRY", "COUNTRY_GRP", "SEX", "YEAR"]:
                flattened_record[f"DIM_{key}"] = value
        
        flattened_records.append(flattened_record)
    
    # Create DataFrame
    df = pd.DataFrame(flattened_records)
    
    # Convert numeric columns to appropriate types
    if not df.empty:
        # Convert YEAR to integer if possible
        df["YEAR"] = pd.to_numeric(df["YEAR"], errors='coerce').astype('Int64')
        
        # Convert VALUE_NUMERIC to float
        df["VALUE_NUMERIC"] = pd.to_numeric(df["VALUE_NUMERIC"], errors='coerce')
        
        # Sort by COUNTRY and YEAR for better organization
        df = df.sort_values(["COUNTRY", "YEAR"], na_position='last')
    
    return df

def get_measure_summary(indicator_code: str, country_groups: List[str] = None, countries: List[str] = None) -> Dict[str, Any]:
    """
    Get a summary of measure data with basic statistics
    
    Args:
        indicator_code: The measure code (e.g., 'hfa_43')
        country_groups: List of country group codes
        countries: List of country ISO codes
    
    Returns:
        Dictionary with summary information
    """
    raw_data = fetch_who_measure_data(indicator_code, country_groups, countries)
    parsed_data = parse_who_data_structure(raw_data)
    
    if not parsed_data:
        return {"error": "No data available", "count": 0}
    
    # Extract basic statistics
    summary = {
        "indicator_code": indicator_code,
        "total_records": len(parsed_data),
        "countries_requested": countries or [],
        "country_groups_requested": country_groups or [],
        "data_sample": parsed_data[:3] if len(parsed_data) > 3 else parsed_data
    }
    
    # Try to extract unique countries from the data
    countries_found = set()
    for record in parsed_data:
        if isinstance(record, dict):
            for key in ["country", "COUNTRY", "country_code", "COUNTRY_CODE"]:
                if key in record:
                    countries_found.add(record[key])
                    break
    
    if countries_found:
        summary["countries_found"] = list(countries_found)
    
    return summary

# Example usage:
# summary = get_measure_summary("hfa_43", ["WHO_EURO"], ["ITA", "MDA"])

In [4]:
# Get raw JSON data
raw_data = fetch_who_measure_data("hfa_43", ["WHO_EURO"], ["ITA", "MDA", "DNK"])

# Parse to DataFrame
df = parse_who_json_to_dataframe(raw_data)

# Now you can work with the data
print(df.head())
print(df.describe())
print(df.groupby('COUNTRY')['VALUE_NUMERIC'].mean())

Establishing session with https://dw.euro.who.int...
API request failed: HTTPSConnectionPool(host='dw.euro.who.int', port=443): Max retries exceeded with url: / (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)')))
Empty DataFrame
Columns: []
Index: []


ValueError: Cannot describe a DataFrame without columns

In [13]:
# Real API Tests - Test with actual WHO API calls
def test_real_api_calls():
    """Test the functions with real API calls for ITA, MDA, DNK"""
    print("=" * 60)
    print("TESTING REAL WHO API CALLS")
    print("=" * 60)
    
    # Test countries
    test_countries = ["ITA", "MDA", "DNK"]
    test_country_groups = ["WHO_EURO", "EU_MEMBERS"]
    test_indicator = "hfa_43"  # Life expectancy at birth
    
    print(f"Testing indicator: {test_indicator}")
    print(f"Testing countries: {test_countries}")
    print(f"Testing country groups: {test_country_groups}")
    print()
    
    # Test 1: Basic API call
    print("1. Testing fetch_who_measure_data()...")
    try:
        data = fetch_who_measure_data(test_indicator, test_country_groups, test_countries)
        print(f"   ✓ API call successful")
        print(f"   ✓ Data type: {type(data)}")
        if isinstance(data, list):
            print(f"   ✓ Number of records: {len(data)}")
            if len(data) > 0:
                print(f"   ✓ Sample record: {data[0]}")
        elif isinstance(data, dict):
            print(f"   ✓ Data keys: {list(data.keys())}")
        else:
            print(f"   ✓ Data: {data}")
    except Exception as e:
        print(f"   ✗ Error: {e}")
    
    print()
    
    # Test 2: Data parsing
    print("2. Testing parse_who_data_structure()...")
    try:
        parsed_data = parse_who_data_structure(data)
        print(f"   ✓ Parsing successful")
        print(f"   ✓ Parsed data type: {type(parsed_data)}")
        if isinstance(parsed_data, list):
            print(f"   ✓ Number of parsed records: {len(parsed_data)}")
        else:
            print(f"   ✓ Parsed data: {parsed_data}")
    except Exception as e:
        print(f"   ✗ Error: {e}")
    
    print()
    
    # Test 3: Summary function
    print("3. Testing get_measure_summary()...")
    try:
        summary = get_measure_summary(test_indicator, test_country_groups, test_countries)
        print(f"   ✓ Summary generated successfully")
        print(f"   ✓ Summary keys: {list(summary.keys())}")
        
        # Display summary details
        for key, value in summary.items():
            if key != "data_sample":
                print(f"   ✓ {key}: {value}")
        
        if "data_sample" in summary and summary["data_sample"]:
            print(f"   ✓ Sample data records: {len(summary['data_sample'])}")
            for i, record in enumerate(summary["data_sample"][:2]):  # Show first 2 records
                print(f"     Record {i+1}: {record}")
    except Exception as e:
        print(f"   ✗ Error: {e}")
    
    print()
    
    # Test 4: Individual country tests
    print("4. Testing individual countries...")
    for country in test_countries:
        print(f"   Testing {country}...")
        try:
            country_data = fetch_who_measure_data(test_indicator, None, [country])
            if isinstance(country_data, list):
                print(f"     ✓ {country}: {len(country_data)} records")
            else:
                print(f"     ✓ {country}: {country_data}")
        except Exception as e:
            print(f"     ✗ {country} Error: {e}")
    
    print()
    print("=" * 60)
    print("REAL API TESTS COMPLETED")
    print("=" * 60)

# Run the real API tests
test_real_api_calls()


TESTING REAL WHO API CALLS
Testing indicator: hfa_43
Testing countries: ['ITA', 'MDA', 'DNK']
Testing country groups: ['WHO_EURO', 'EU_MEMBERS']

1. Testing fetch_who_measure_data()...
Establishing session with https://dw.euro.who.int...
Making API request to https://dw.euro.who.int/api/v3/measures/hfa_43?filter=COUNTRY_GRP:WHO_EURO,EU_MEMBERS;COUNTRY:ITA,MDA,DNK...
   ✓ API call successful
   ✓ Data type: <class 'list'>
   ✓ Number of records: 244
   ✓ Sample record: {'fact_id': '52996984', 'attributes': {'MEASURE_TYPE': 'AVG_ARITH'}, 'dimensions': {'COUNTRY': 'DNK', 'COUNTRY_GRP': '', 'SEX': 'ALL', 'YEAR': '1969'}, 'value': {'display': '73.3', 'numeric': 73.34}}

2. Testing parse_who_data_structure()...
   ✓ Parsing successful
   ✓ Parsed data type: <class 'list'>
   ✓ Number of parsed records: 244

3. Testing get_measure_summary()...
Establishing session with https://dw.euro.who.int...
Making API request to https://dw.euro.who.int/api/v3/measures/hfa_43?filter=COUNTRY_GRP:WHO_EURO,E